In [1]:
# 원래는... 평가 기준이 6개 중에 정답이 top 5안에 들었냐?(Reall@5)인데
# collate_fn_with_neg에서 뽑은 neg 5개로 평가를 하니... pos 1개, neg 5개.. 즉 찍어도 5/6 = 83퍼가 나옴
# => 긍까 평가 로직을.... 전체 책 중에서 내 정답을 찾을 수 있는지.. Full Ranking 테스트로 바꿈

In [ ]:
# 지금 SeqRec이나 RecGPT는 Retrieval 모델임! User Emb를 만들어서 DB에서 후보를 긁어오기 위함!
# 따라서 Retrieval 모델의 성능은 @20이나 @50에서 얼마나 정답을 잘 건져오는지(Recall)가 핵심이래

# Retrieval 모델에서 막 100개씩 뽑아놨다고 하면... 이제 랭킹을 해야하는데...
# 얘는 유저의 정보도 더 섬세하고... 클랙, 구매 등등 활동도 세밀하게 보면서 1등을 뽑아내는 느낌임!!
# 1. LightGBM / XGBoost: 머신러닝의 클래식. 빠르고 성능 좋아서 현업에서 엄청 많이 씀
# 2. DCN (Deep Cross Network): 딥러닝 기반. 특성(Feature)들을 교차시켜서 복잡한 패턴을 찾음
# 3. DIN (Deep Interest Network): 유저가 전에 본 상품 중 현재 후보 아이템과 관련된 것에만 Attention
# 얘네는 서비스 런칭 후에... 유저가 클릭은 하는데 구매를 안하네? 같은 정교한 목표가 생기면 하셈

In [2]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup
from transformers import GPT2Model

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)            # 기본 Python random 고정
    np.random.seed(seed)         # NumPy 랜덤 고정
    torch.manual_seed(seed)      # CPU 연산 랜덤 고정
    torch.cuda.manual_seed(seed) # GPU 모든 디바이스 랜덤 고정
    torch.cuda.manual_seed_all(seed)  # 멀티 GPU일 때

    # 연산 재현성
    torch.backends.cudnn.deterministic = True  # cuDNN 연산을 determinisitc으로 강제
    torch.backends.cudnn.benchmark = False     # CUDA 성능 자동 튜닝 기능 끔 → 완전 재현 가능
    
set_seed(42)

In [3]:
class RecGPTWrapper(nn.Module):
    def __init__(self, e5_dim):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")
        hidden_dim = self.gpt2.config.hidden_size  # 768

        self.input_proj = nn.Linear(e5_dim, hidden_dim)
        self.user_proj = nn.Linear(hidden_dim, e5_dim)

    def forward(self, seq_embs, seq_mask=None):
        x = self.input_proj(seq_embs)  # [B, L, 768]
        out = self.gpt2(inputs_embeds=x, attention_mask=seq_mask).last_hidden_state
        user_emb = self.user_proj(out[:, -1])  # [B, e5_dim]
        return F.normalize(user_emb, dim=-1)

In [4]:
# =========================================
# 0️⃣ 기본 설정
# =========================================
PAD_ID = -1 # 모델에서 padding_idx랑 일치시켜야 함
EMB_DIM = 384  # E5 임베딩 차원

# book_emb_dict에 PAD_ID embedding 추가
book_emb_dict = np.load("book_emb_dict.npy", allow_pickle=True).item()
book_emb_dict[PAD_ID] = np.zeros(EMB_DIM, dtype=np.float32)

DATA_PATH = './data/user_book_sequence.parquet'
df_user = pd.read_parquet(DATA_PATH)

# 시퀀스 전처리: 없는 책은 PAD_ID로 대체
df_user["book_seq_padded"] = df_user["book_seq"].apply(
    lambda seq: [b if b in book_emb_dict else PAD_ID for b in seq]
)

"""
SASRec, BERT4Rec, GRU4Rec, 이런 시퀀스 기반 모델들은
기본적으로 “임베딩 테이블(index 기반)”을 입력으로 받음

즉, embedding_table[i] → i번째 아이템의 벡터
각 아이템(Book)을 “정수 index”로 표현해야 함
"""
# SASRec 입력용 index 변환을 위한 매핑
book_ids = sorted(book_emb_dict.keys())
book2idx = {b: i for i, b in enumerate(book_ids)}
idx2book = {i: b for i, b in enumerate(book_ids)}

# torch 텐서로 변환
book_emb_matrix = torch.tensor(
    np.stack([book_emb_dict[b] for b in book_ids]),
    dtype=torch.float32  # float16로 메모리 절감은 나중에
)  # (num_books, 384)

# dict 삭제 (RAM 확보)
del book_emb_dict
import gc; gc.collect()

0

In [5]:
# =========================================
# user sequences & positive book embeddings
# ① 유저별 시퀀스를 인덱스로 바꾸고
# ② 마지막 책을 positive target으로 뽑은 다음
# ③ 그 책들의 임베딩(pos_book_embs)을 텐서로 만드는 것
# =========================================
max_len = 100
user_seqs_list = []
pos_book_ids_list = []

for seq in tqdm(df_user["book_seq_padded"], desc="Making Idx..."):
    seq = list(seq)
    if len(seq) == 0:
        user_seqs_list.append([book2idx[PAD_ID]] * max_len)
        pos_book_ids_list.append(book2idx[PAD_ID])
        continue

    pos_book_id = seq[-1]
    pos_book_ids_list.append(book2idx.get(pos_book_id, book2idx[PAD_ID]))

    pad_len = max(0, max_len - len(seq))
    seq_idx = [book2idx.get(b, book2idx[PAD_ID]) for b in seq[-max_len:]]
    seq_idx = [book2idx[PAD_ID]] * pad_len + seq_idx
    user_seqs_list.append(seq_idx)

# 텐서 변환
user_seqs_tensor = torch.tensor(user_seqs_list, dtype=torch.long)
pos_book_ids_tensor = torch.tensor(pos_book_ids_list, dtype=torch.long)

Making Idx...: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 836433/836433 [00:19<00:00, 43110.23it/s]


In [6]:
total_len = user_seqs_tensor.size(0)
subset_len = int(total_len * 0.3)

# 1) 랜덤 인덱스 생성 (순서를 섞음)
rand_indices = torch.randperm(total_len) 

# 2) 앞쪽 50% 인덱스만 가져옴
selected_indices = rand_indices[:subset_len]

# 3) 텐서 자르기 (Fancy Indexing)
user_seqs_tensor = user_seqs_tensor[selected_indices]
pos_book_ids_tensor = pos_book_ids_tensor[selected_indices]

print(f"데이터 다운샘플링 완료: {total_len} -> {len(user_seqs_tensor)} 개 사용")

데이터 다운샘플링 완료: 836433 -> 250929 개 사용


In [7]:
"""
Offline Negative Sampling
→ “샘플링 방식이 복잡하거나 비모델 의존적일 때” (예: popularity-based, sequence-aware 등)
→ 즉, 학습 중에 계산하기엔 너무 무거울 때

고정된 Negative Pool을 쓰는 경우
→ 학습 중에 랜덤이 아니라, 특정 negative 후보군을 유지하고 싶을 때

이럴때만 dataset에서 neg sampling을 하니까 일단은 빼도록 하자

# **neg sampling**
# 같은 pos pool에서 랜덤 선택 ㄴㄴ
# 전체 `book_emb_matrix`에서 샘플링 (훨씬 일반적) 하도록 해야하는데 나중에..
"""
class UserBookDataset(Dataset):
    def __init__(self, user_seqs, pos_book_ids, book_emb_matrix):
        """
        user_seqs: list or tensor (num_users, seq_len)
        pos_book_ids: tensor (num_users,)
        book_emb_matrix: torch.Tensor (num_books, emb_dim)
        """
        self.user_seqs = user_seqs
        
        # self.pos_book_embs = pos_book_embs -> init에서 직접 들고 있으면 메모리 터짐
        # `pos_book_ids`만 저장하고, 임베딩은 on-demand로 가져옴 |
        self.pos_book_ids = pos_book_ids

        self.book_emb_matrix = book_emb_matrix
        self.num_books = book_emb_matrix.size(0)
    
    def __len__(self):
        return len(self.user_seqs)

    def __getitem__(self, idx):
        # user sequence
        ''' 
        user_seq = torch.tensor(self.user_seqs[idx], dtype=torch.long)
        `user_seqs`가 이미 tensor면 그대로 사용 가능??????? 
        '''
        user_seq = self.user_seqs[idx]

        # positive book embedding
        pos_id = self.pos_book_ids[idx]
        # pos_emb = self.book_emb_matrix[pos_id]
        # 이거.... pos_emb를 가져오면서 collate_fn이 하는 역할을 침범하고 있대
        # collate_fn에서 batch 단위에서 pos/neg 임베딩 변환을 처리해야 하는데..

        return user_seq, pos_id

In [8]:
def collate_fn_with_neg(batch, book_emb_matrix, n_neg=5, hard_ratio=0.5, pad_idx=None):
    """
    batch: list of (user_seq, pos_id)
    book_emb_matrix: [num_books, emb_dim]
    n_neg: negative sample 개수
    hard_ratio: in-batch hard negative 비율
    """
    user_seqs_idx, pos_ids = zip(*batch)
    # ensure tensors
    user_seqs_idx = torch.stack(user_seqs_idx).long()  # [B, L]
    pos_ids = torch.stack(pos_ids).long()              # [B]
    B, L = user_seqs_idx.shape

    # Embedding lookup: prefer nn.Embedding.from_pretrained(book_emb_matrix) and use indices
    # If book_emb_matrix is tensor passed here, indexing is OK but costly if then .to(device)
    user_seq_embs = book_emb_matrix[user_seqs_idx]  # [B, L, D]
    pos_embs = book_emb_matrix[pos_ids]             # [B, D]

    # negatives
    n_hard = int(n_neg * hard_ratio)
    n_rand = n_neg - n_hard
    neg_embs_list = []
    
    if n_hard > 0:
        # perm guaranteed non-equal by rotate
        perm = (torch.arange(B) + 1) % B
        hard_neg = pos_embs[perm][:, None, :].repeat(1, n_hard, 1)  # [B, n_hard, D]
        neg_embs_list.append(hard_neg)
        
    if n_rand > 0:
        num_books = book_emb_matrix.size(0)

        # 이건 완전 랜덤 샘플링... 사용자가 이미 본 책이나 pos item이 neg로 들어갈 수도 있음
        rand_ids = torch.randint(0, num_books, (B, n_rand), dtype=torch.long)
        
        # rand_ids[rand_ids >= pos_ids.unsqueeze(1)] += 1 # positive만 제외하고 negative를 샘플링
        rand_ids = torch.randint(0, num_books - 1, (B, n_rand))
        pos = pos_ids.unsqueeze(1)
        rand_ids = rand_ids + (rand_ids >= pos).long()

        rand_neg = book_emb_matrix[rand_ids]  # [B, n_rand, D]

        # # “positive item이나 이미 본 책이 negative로 들어가는” 경우 지워
        # # 근데 이걸로 바꾸니까 개개개개 느려지네 ㄷㄷ
        # neg_embs_users = []
        # for i in range(B):
        #     # 제외할 책 목록: 사용자가 이미 본 책들 + 현재 positive
        #     exclude_ids = set(user_seqs_idx[i].tolist() + [pos_ids[i].item()])
        #     candidates = list(set(range(num_books)) - exclude_ids)
        #     rand_neg_ids = random.sample(candidates, n_rand)
        #     neg_embs_users.append(book_emb_matrix[torch.tensor(rand_neg_ids)])

        # rand_neg = torch.stack(neg_embs_users, dim=0)  # [B, n_rand, D]
        
        neg_embs_list.append(rand_neg)

    neg_embs = torch.cat(neg_embs_list, dim=1)  # [B, n_neg, D]
    return user_seq_embs, pos_embs, neg_embs, user_seqs_idx

In [ ]:
# import torch
# import torch.nn.functional as F
# import random

# def collate_fn_semantic_neg(batch, 
#                             book_emb_matrix, 
#                             book_clusters,      # dict {book_id: cluster_id}
#                             cluster_to_books,   # dict {cluster_id: list of book_ids}
#                             book_pop_count,     # dict {book_id: read_count}
#                             n_neg=5, 
#                             top_k_sim=50, 
#                             hard_ratio=0.3, 
#                             popularity_ratio=0.2,
#                             pad_idx=None):
#     """
#     batch: list of (user_seq_tensor, pos_id)
#     book_emb_matrix: [num_books, D] tensor
#     book_clusters: book_id -> cluster_id
#     cluster_to_books: cluster_id -> list of book_ids
#     book_pop_count: book_id -> read_count
#     n_neg: negative 개수
#     top_k_sim: cluster 내 top-k 후보에서 랜덤 선택
#     hard_ratio: in-batch hard negative 비율
#     popularity_ratio: popularity negative 비율
#     """
#     user_seqs_idx, pos_ids = zip(*batch)
#     user_seqs_idx = torch.stack(user_seqs_idx).long()
#     pos_ids = torch.stack(pos_ids).long()
#     B, L = user_seqs_idx.shape
#     D = book_emb_matrix.size(1)

#     # --- user & pos embeddings ---
#     user_seq_embs = book_emb_matrix[user_seqs_idx]       # [B, L, D]
#     pos_embs = book_emb_matrix[pos_ids]                 # [B, D]

#     # --- negative sampling ---
#     n_hard = int(n_neg * hard_ratio)
#     n_pop = int(n_neg * popularity_ratio)
#     n_sem = n_neg - n_hard - n_pop
#     neg_embs_list = []

#     # (1) In-batch hard negative
#     if n_hard > 0:
#         perm = (torch.arange(B) + 1) % B
#         hard_neg = pos_embs[perm][:, None, :].repeat(1, n_hard, 1)  # [B, n_hard, D]
#         neg_embs_list.append(hard_neg)

#     # (2) Popularity-aware negative
#     if n_pop > 0:
#         top_pop_books = sorted(book_pop_count, key=book_pop_count.get, reverse=True)
#         top_pop_books = top_pop_books[:int(len(top_pop_books)*0.05)]  # 상위 5%
#         pop_neg_ids = []
#         for i in range(B):
#             candidates = list(set(top_pop_books) - set(user_seqs_idx[i].tolist()) - {pos_ids[i].item()})
#             if len(candidates) < n_pop:
#                 candidates = top_pop_books  # fallback
#             pop_neg_ids.append(random.sample(candidates, n_pop))
#         pop_neg_ids = torch.tensor(pop_neg_ids, dtype=torch.long)  # [B, n_pop]
#         pop_neg_embs = book_emb_matrix[pop_neg_ids]                 # [B, n_pop, D]
#         neg_embs_list.append(pop_neg_embs)

#     # (3) Semantic / cluster-based negative
#     if n_sem > 0:
#         sem_neg_ids = []
#         for i in range(B):
#             pos_book_id = pos_ids[i].item()
#             cluster_id = book_clusters[pos_book_id]
#             cluster_books = cluster_to_books[cluster_id]
#             cluster_books = list(set(cluster_books) - set(user_seqs_idx[i].tolist()) - {pos_book_id})
#             if len(cluster_books) == 0:
#                 cluster_books = list(range(book_emb_matrix.size(0)))  # fallback
#             # cluster 내 embeddings
#             candidate_embs = book_emb_matrix[cluster_books]
#             pos_emb = pos_embs[i]
#             cos_sim = F.cosine_similarity(pos_emb.unsqueeze(0), candidate_embs, dim=-1)
#             top_candidates = torch.topk(-cos_sim, min(top_k_sim, len(cos_sim))).indices
#             selected = random.sample(top_candidates.tolist(), n_sem)
#             sem_neg_ids.append([cluster_books[s] for s in selected])
#         sem_neg_ids = torch.tensor(sem_neg_ids, dtype=torch.long)  # [B, n_sem]
#         sem_neg_embs = book_emb_matrix[sem_neg_ids]                 # [B, n_sem, D]
#         neg_embs_list.append(sem_neg_embs)

#     # concat all negatives
#     neg_embs = torch.cat(neg_embs_list, dim=1)  # [B, n_neg, D]

#     return user_seq_embs, pos_embs, neg_embs, user_seqs_idx

In [9]:
# ===============================
# Hyperparameters
# ===============================
batch_size = 128
e5_dim = 384
hidden_dim = 128
proj_dim = 128
n_neg = 5
lr = 1e-3
weight_decay=1e-5
tau = 0.07
epochs = 20
min_len = 5 # 이거 지피티한테 이따가 물어보자.
# 그그 막 3.8만개 시퀀스 정보가 있는 애들도 있어서 걔네를... 뭐 어케 짤라서 데이터를 더 생성할지...
# 평균은 134이고 중간값은 59정도니까 뭐 얼마 이상인 애들만 제대로 학습되도록 해야할지 그런거 좀 정해야할거같음

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
# user별로 시퀀스를 잘라서 다시 데이터를 구성
train_user_seqs = []
valid_user_seqs = []
train_pos_ids = []
valid_pos_ids = []

for seq in user_seqs_tensor:
    seq = seq.tolist() if torch.is_tensor(seq) else seq
    if len(seq) < min_len:
        continue 

    # ✅ Valid: 가장 마지막 아이템을 맞추기 (Leave-One-Out)
    # 입력: 처음 ~ 마지막-1 / 정답: 마지막
    valid_user_seqs.append(torch.tensor(seq[:-1])) 
    valid_pos_ids.append(torch.tensor(seq[-1]))

    # ✅ Train: Valid에 쓰인 '마지막 아이템'은 절대 보여주면 안 됨!
    # 입력: 처음 ~ 마지막-2 / 정답: 마지막-1
    # 이렇게 해야 Train 데이터가 Valid의 정답(마지막 아이템)을 모릅니다.
    train_user_seqs.append(torch.tensor(seq[:-2]))
    train_pos_ids.append(torch.tensor(seq[-2]))

In [11]:
from torch.nn.utils.rnn import pad_sequence

# train_user_seqs = torch.stack(train_user_seqs)
# 이 코드는 모든 유저의 시퀀스 길이(len(seq))가 똑같을 때만 작동합니다. 
# 유저마다 읽은 책 개수가 다르면 (seq 길이가 제각각이면) RuntimeError가 납니다.

# 0으로 패딩 (batch_first=True로 해야 [Batch, Max_Len] 모양이 됨)
train_user_seqs = pad_sequence(train_user_seqs, batch_first=True, padding_value=0)
valid_user_seqs = pad_sequence(valid_user_seqs, batch_first=True, padding_value=0)

# Pos IDs는 스칼라(숫자 하나)들이므로 그냥 stack 해도 됨
train_pos_ids = torch.stack(train_pos_ids)
valid_pos_ids = torch.stack(valid_pos_ids)

In [12]:
train_dataset = UserBookDataset(train_user_seqs, train_pos_ids, book_emb_matrix)
valid_dataset = UserBookDataset(valid_user_seqs, valid_pos_ids, book_emb_matrix)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          collate_fn=lambda b: collate_fn_with_neg(b, book_emb_matrix, n_neg=5)
)

valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False,
                          collate_fn=lambda b: collate_fn_with_neg(b, book_emb_matrix, n_neg=5)
)

In [13]:
def evaluate_full_ranking(model, data_loader, all_book_embs, device, k_list=[5, 10]):
    """
    모든 책(all_book_embs)을 후보로 두고 평가 (Full Ranking)
    """
    model.eval()
    all_book_embs = all_book_embs.to(device)  # [Total_Books, D]
    # 모든 책 미리 Normalize (내적 = 코사인 유사도)
    all_book_embs = F.normalize(all_book_embs, dim=-1)
    
    metrics = {f'Recall@{k}': 0.0 for k in k_list}
    metrics.update({f'NDCG@{k}': 0.0 for k in k_list})
    metrics['MRR'] = 0.0
    total_count = 0
    
    with torch.no_grad():
        for user_seqs_idx, pos_ids in data_loader:# tqdm(data_loader, desc=f"Eval: "):
            # Validation Loader는 Negative 필요 없음 (user_seq, pos_id)만 있으면 됨
            user_seqs_idx = user_seqs_idx.to(device) # [Batch, Seq_Len]
            pos_ids = pos_ids.to(device)             # [Batch]
            
            # 1. E5 임베딩 Lookup (book_emb_matrix는 CPU에 있어도 됨, 여기서 필요한 것만)
            # 주의: model 안에 임베딩 레이어가 없다면 밖에서 변환해서 넣어줘야 함
            # 여기서는 편의상 user_seq_embs가 이미 만들어져 있다고 가정하거나 lookup
            # (님 코드 맥락에 맞춰 수정 필요)
            user_seq_embs = all_book_embs[user_seqs_idx] # [B, L, D] (이미 normalized)
            
            # 2. 유저 벡터 생성
            # Masking: 패딩(0)인 부분 처리
            seq_mask = (user_seqs_idx != 0) 
            user_emb = model(user_seq_embs) # SASRec 통과 [B, D] (마지막 시점)
            user_emb = F.normalize(user_emb, dim=-1)
            
            # 3. 전체 책과 유사도 계산 (행렬곱)
            # scores: [Batch, Total_Books]
            scores = torch.matmul(user_emb, all_book_embs.T)
            
            # 4. 이미 읽은 책(History) 마스킹 (선택 사항)
            # 유저가 과거에 읽은 책은 추천 후보에서 제외하고 싶다면:
            # rows = torch.arange(user_seqs_idx.size(0)).view(-1, 1)
            # scores[rows, user_seqs_idx] = -9999.0
            
            # 5. Top-K 추출 (전체 정렬은 느리니까 topk 사용)
            max_k = max(k_list)
            _, topk_indices = torch.topk(scores, k=max_k, dim=1) # [B, max_k]
            
            # 6. 정답 확인 및 메트릭 계산
            for i in range(len(pos_ids)):
                target = pos_ids[i].item()
                pred_list = topk_indices[i].tolist()
                
                # MRR (Top-K 내에 없으면 0 처리 or 전체 랭킹 계산 필요하지만 보통 Top-K MRR 씀)
                if target in pred_list:
                    rank = pred_list.index(target) + 1
                    metrics['MRR'] += 1.0 / rank
                
                for k in k_list:
                    if target in pred_list[:k]:
                        metrics[f'Recall@{k}'] += 1.0
                        metrics[f'NDCG@{k}'] += 1.0 / torch.log2(torch.tensor(pred_list.index(target) + 2.0)).item()
            
            total_count += len(pos_ids)

    # 평균 내기
    return {k: v / total_count for k, v in metrics.items()}

In [14]:
# 검증(Validation) 때는 Negative Sample을 굳이 만들 필요가 없습니다. 정답만 딱 리턴하는 함수를 하나 만드세요.
def collate_fn_eval(batch):
    """
    평가 때는 Negative도 필요 없고, Embedding 변환도 필요 없음.
    오직 정수 ID (Sequence, Target)만 리턴.
    """
    # batch = list of (user_seq, pos_id, ...) <- 데이터셋이 뭘 뱉든
    # 우리는 앞의 두 개(user_seq, pos_id)만 필요함!
    user_seqs = [item[0] for item in batch] # item[0]이 이미 텐서이므로 그냥 가져옴
    user_seqs = torch.stack(user_seqs).long()
    
    pos_ids = [item[1] for item in batch] # item[1]도 이미 텐서이므로 그냥 가져옴
    pos_ids = torch.stack(pos_ids).long()
    
    return user_seqs, pos_ids

eval_loader = DataLoader(
    valid_dataset, 
    batch_size=128, # 메모리 허용하면 더 크게 해도 됨
    collate_fn=collate_fn_eval,
    shuffle=False
)

In [15]:
import wandb

wandb.init(
    project="sequence-recommendation", 
    name="recgpt-e5-lora-distill",
    config={
        # "lr": LEARNING_RATE,
        # "temperature": 0.05,
        "epochs": 20,
        "batch_size": train_loader.batch_size,
    },
    mode="online"  # 오프라인 방지가 필요할 때
)

/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

In [16]:
# 🔹 2) 모델 초기화
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# recgpt = RecGPTWrapper(e5_dim=e5_dim, hidden_dim=768).to(device)
recgpt = RecGPTWrapper(e5_dim=e5_dim).to(device)

optimizer = optim.AdamW(recgpt.parameters(), lr=lr, weight_decay=weight_decay)
num_training_steps = len(train_loader) * epochs
num_warmup_steps = int(num_training_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

wandb.watch(recgpt, log="all")  # gradients, parameters 기록

In [17]:
# 🔹 3) 학습 루프
for epoch in range(epochs):
    recgpt.train()
    total_loss = 0.0

    for user_seq_embs, pos_book_embs, neg_book_embs, user_seqs_idx in tqdm(train_loader, desc=f"[Epoch: {epoch+1}] Train: "):
        user_seq_embs = user_seq_embs.to(device)
        pos_book_embs = pos_book_embs.to(device)
        neg_book_embs = neg_book_embs.to(device)
        seq_mask = (user_seqs_idx != 0).to(device)  # 패딩 마스크 예시

        # 🔹 (1) user embedding
        user_emb = recgpt(user_seq_embs, seq_mask=seq_mask)  # [B, H]

        # 🔹 (2) normalize
        user_emb = F.normalize(user_emb, dim=-1)
        pos_book_embs = F.normalize(pos_book_embs, dim=-1)
        neg_book_embs = F.normalize(neg_book_embs, dim=-1)

        # 🔹 (3) InfoNCE Loss
        pos_sim = torch.sum(user_emb * pos_book_embs, dim=-1) / tau      # [B]
        neg_sim = torch.einsum('bd,bnd->bn', user_emb, neg_book_embs) / tau  # [B, n_neg]
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
        labels = torch.zeros(len(pos_sim), dtype=torch.long, device=user_emb.device)
        loss = F.cross_entropy(logits, labels)

        # 🔹 (4) Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * user_seq_embs.size(0)

    avg_train_loss = total_loss / len(train_loader.dataset)

    # 🔹 5) Validation
    recgpt.eval()
    total_valid_loss = 0.0
    with torch.no_grad():
        for user_seq_embs, pos_book_embs, neg_book_embs, user_seqs_idx in tqdm(valid_loader, desc=f"[Epoch: {epoch+1}] Valid: "):
            user_seq_embs = user_seq_embs.to(device)
            pos_book_embs = pos_book_embs.to(device)
            neg_book_embs = neg_book_embs.to(device)
            seq_mask = (user_seqs_idx != 0).to(device)

            user_emb = recgpt(user_seq_embs, seq_mask=seq_mask)
            user_emb = F.normalize(user_emb, dim=-1)
            pos_book_embs = F.normalize(pos_book_embs, dim=-1)
            neg_book_embs = F.normalize(neg_book_embs, dim=-1)

            pos_sim = torch.sum(user_emb * pos_book_embs, dim=-1) / tau
            neg_sim = torch.einsum('bd,bnd->bn', user_emb, neg_book_embs) / tau
            logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
            labels = torch.zeros(len(pos_sim), dtype=torch.long, device=user_emb.device)
            loss = F.cross_entropy(logits, labels)

            total_valid_loss += loss.item() * user_seq_embs.size(0)
            
    avg_valid_loss = total_valid_loss / len(valid_loader.dataset)
    print(f"[Epoch {epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_valid_loss:.4f}")

    # [Step B] 진짜 평가: Full Ranking Evaluation (새 함수 사용)
    # 주의: all_book_embs는 전체 책 임베딩 텐서여야 함 [Num_Books, Emb_Dim]
    print("Running Full Ranking Evaluation...")
    real_metrics = evaluate_full_ranking(
        model=recgpt,
        data_loader=eval_loader, # 위에 만든 간단한 로더
        all_book_embs=book_emb_matrix,   # 전체 책 임베딩 (CPU에 있어도 함수 안에서 GPU로 보냄)
        device=device,
        k_list=[5, 10]
    )
    print(f"Recall@5={real_metrics['Recall@5']:.4f}, NDCG@5={real_metrics['NDCG@5']:.4f}, MRR={real_metrics['MRR']:.4f}")
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "valid_loss": avg_valid_loss,
        "valid/recall@5": real_metrics["Recall@5"],
        "valid/ndcg@5": real_metrics["NDCG@5"],
        "valid/mrr": real_metrics["MRR"],
        "lr": scheduler.get_last_lr()[0],
    })

[Epoch: 1] Valid: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1961/1961 [03:04<00:00, 10.61it/s]


[Epoch 1/20] Train Loss: 1.3423 | Valid Loss: 1.2641
Running Full Ranking Evaluation...
Recall@5=0.0003, NDCG@5=0.0002, MRR=0.0002


[Epoch: 2] Valid: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1961/1961 [03:06<00:00, 10.52it/s]


[Epoch 2/20] Train Loss: 1.1744 | Valid Loss: 1.2308
Running Full Ranking Evaluation...
Recall@5=0.0002, NDCG@5=0.0001, MRR=0.0001


[Epoch: 3] Valid: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1961/1961 [03:04<00:00, 10.60it/s]


...

[Epoch 18/20] Train Loss: 1.0325 | Valid Loss: 1.1611
Running Full Ranking Evaluation...
Recall@5=0.0016, NDCG@5=0.0010, MRR=0.0009


[Epoch: 19] Valid: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1961/1961 [03:04<00:00, 10.62it/s]


[Epoch 19/20] Train Loss: 1.0290 | Valid Loss: 1.1604
Running Full Ranking Evaluation...
Recall@5=0.0016, NDCG@5=0.0010, MRR=0.0009


[Epoch: 20] Valid: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1961/1961 [03:04<00:00, 10.62it/s]


[Epoch 20/20] Train Loss: 1.0281 | Valid Loss: 1.1603
Running Full Ranking Evaluation...
Recall@5=0.0016, NDCG@5=0.0010, MRR=0.0009
